# Explore Sedona corridor zonal statistics

Earth Engine **reduceRegions** analogue with Apache Sedona:

- Load a synthetic NDVI GeoTIFF (`RS_FromGeoTiff`)
- Buffer a transmission line in EPSG:32610
- Compute `RS_ZonalStatsAll` over the corridor
- Inspect `explain()` for native raster SQL

In [ ]:
import json
import tempfile
from pathlib import Path

import numpy as np
import rioxarray  # noqa: F401
import xarray as xr
from pyproj import Transformer
from pyspark.sql import functions as F
from rasterio.transform import from_origin
from shapely.geometry import LineString, mapping

from wildfire_geo_ml.features.indices import write_scene_index_cogs
from wildfire_geo_ml.sedona.session import create_sedona_session
from wildfire_geo_ml.sedona.zonal_stats_job import load_corridor_regions, stop_spark

In [ ]:
workdir = Path(tempfile.mkdtemp(prefix="sedona_notebook_"))
indices_dir = workdir / "indices" / "DEMO_SCENE"
indices_dir.mkdir(parents=True)

values = np.full((32, 32), 0.55, dtype=np.float32)
transform = from_origin(600000.0, 4500000.0, 30.0, 30.0)
ndvi = xr.DataArray(values, dims=("y", "x"))
ndvi = ndvi.rio.write_crs("EPSG:32610")
ndvi = ndvi.rio.write_transform(transform)
write_scene_index_cogs({"ndvi": ndvi}, workdir / "indices", "DEMO_SCENE")
cog_path = indices_dir / "ndvi.tif"
cog_path

In [ ]:
lines_path = workdir / "lines.geojson"
transformer = Transformer.from_crs("EPSG:32610", "EPSG:4326", always_xy=True)
line_utm = LineString([(599500.0, 4499500.0), (601000.0, 4500500.0)])
coords = [transformer.transform(x, y) for x, y in line_utm.coords]
payload = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": mapping(LineString(coords)),
            "properties": {"OBJECTID": 1, "TLine Name": "DEMO_LINE"},
        }
    ],
}
lines_path.write_text(json.dumps(payload), encoding="utf-8")
lines_path

In [ ]:
spark = create_sedona_session(app_name="notebook-sedona-zonal")
binary_df = spark.read.format("binaryFile").load(str(cog_path))
rasters_df = binary_df.withColumn("raster", F.expr("RS_FromGeoTiff(content)"))
regions_df = load_corridor_regions(
    spark,
    lines_path,
    buffer_m=100.0,
    metric_crs="EPSG:32610",
)
joined = rasters_df.crossJoin(F.broadcast(regions_df))
stats_df = joined.withColumn(
    "stats",
    F.expr("RS_ZonalStatsAll(raster, geometry, 1, true, true)"),
).filter(F.col("stats").isNotNull())
stats_df.selectExpr(
    "region_id",
    "region_kind",
    "stats.mean as ndvi_mean",
    "stats.stddev as ndvi_std",
).show(truncate=False)

In [ ]:
plan = stats_df._jdf.queryExecution().toString()  # noqa: SLF001
assert "CartesianProduct" not in plan
print(plan)
stop_spark(spark)